# Build a ReAct Agent with LangGraph on Flyte

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/starter-examples/langgraph-react-agent/tutorial_langgraph_react_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Build a simple ReAct (Reason + Act) agent using LangGraph's prebuilt `create_react_agent` and run it on Flyte.

**What you'll learn:**
- Define tools with `@tool` and `@flyte.trace` for observability
- Use LangGraph's prebuilt ReAct agent
- Run the agent as a Flyte task locally and remotely

**Pattern:**
```
User: "What is 12 * 7 plus 3?"
  → Agent reasons: need to multiply first
  → Calls multiply(12, 7) → 84
  → Reasons: now add 3
  → Calls add(84, 3) → 87
  → Returns: "87"
```

---

## Setup

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/starter-examples/langgraph-react-agent/
    !pip install -r requirements.txt

from utils.file_viewer import view_file

## Dependencies

The example uses LangGraph, LangChain, and Flyte:

In [8]:
view_file("requirements.txt")

## Set your API Key

The agent uses OpenAI, so you'll need an API key.

You can set it as an environment variable or enter it below:

In [9]:
import os
from getpass import getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on a remote Flyte cluster, add the API key as a secret:

```bash
flyte create secret OPENAI_API_KEY
```

## Connect to Flyte Cluster

Skip this step if you only want to run locally.

- Don't have a Flyte cluster? Request access at [flyte.org](https://flyte.org/)
- Already have one? Set your endpoint below:

In [5]:
!flyte create config \
    --endpoint <your-endpoint> \
    --auth-type headless \
    --builder remote \
    --domain development \
    --project flytesnacks

zsh:1: no such file or directory: your-endpoint


---

## Code Walkthrough

The entire agent fits in a single file. Let's walk through it:

In [4]:
view_file("langgraph_react_agent.py")

### Key Components

**1. Flyte Environment** — Defines the container image, secrets, and resources for the task.

```python
env = flyte.TaskEnvironment(
    name="langgraph_env",
    image=flyte.Image.from_debian_base().with_requirements("requirements.txt"),
    secrets=[flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")],
)
```

**2. Tools** — Simple Python functions decorated with `@tool` (LangChain) and `@flyte.trace` (observability). Must be `async`.

```python
@tool
@flyte.trace
async def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b
```

**3. Agent Task** — Uses LangGraph's `create_react_agent` to wire up the LLM + tools into a ReAct loop, wrapped as a Flyte task.

```python
@env.task
async def agent(request: str) -> str:
    llm = ChatOpenAI(model="gpt-4o-mini")
    react_agent = create_react_agent(llm, tools)
    result = await react_agent.ainvoke(...)
```

---

## Run the Agent

### Local run

In [10]:
!uv run flyte run --local langgraph_react_agent.py agent --request "What is 12 * 7 plus 3?"

⠹ Launching local execution...0m15:33:29.871210 ERROR    taskrunner.py:104 -                                    
                         [5e1c92b1-f7ae-45d8-82e3-c9da7169dca7][4x0lb0lvlp4ipv8t
                         k1zbmvj08]  Task failed with error: Error code: 401 -  
                         {'error': {'message': "You didn't provide an API key.  
                         You need to provide your API key in an Authorization   
                         header using Bearer auth (i.e. Authorization: Bearer   
                         YOUR_KEY), or as the password field (with blank        
                         username) if you're accessing the API from your browser
                         and are prompted for a username and password. You can  
                         obtain an API key from                                 
                         https://platform.openai.com/account/api-keys.", 'type':
                         'invalid_request_error', 'param': None, 'code': None

### Remote run

The first run will build and push a container image, which may take a few minutes.

In [ ]:
!uv run flyte run langgraph_react_agent.py agent --request "What is 12 * 7 plus 3?"

### Try different prompts

In [ ]:
!uv run flyte run --local langgraph_react_agent.py agent --request "Multiply 15 by 4 and then add 20"

In [ ]:
!uv run flyte run --local langgraph_react_agent.py agent --request "What is 100 divided by 4 times 3?"

---

## Key Takeaways

- **LangGraph + Flyte** — Use LangGraph for agent logic, Flyte for orchestration, scaling, and observability
- **`@flyte.trace`** — Gives you visibility into each tool call in the Flyte dashboard
- **`create_react_agent`** — LangGraph's prebuilt ReAct loop handles the reasoning cycle for you
- **Single file** — The entire agent is self-contained in one `.py` file

## Next Steps

- Add more tools (web search, file I/O, database queries)
- Try different LLMs (`gpt-4o`, `claude-sonnet-4-5-20250929`)
- Add a Flyte report to visualize the agent's reasoning trace
- Check out the [multi-agent tutorials](../../multi-agent-workflows/) for more complex patterns